# E-Commerce Sales & Customer Behavior Analysis

This notebook runs the production modules in `src/` and provides an analyst-friendly walkthrough for portfolio reviewers.

## 1. Load and Clean Data

Place the nine Olist CSV files in `data/raw` before running the notebook.

In [ ]:
from src.config import DATA_RAW, DATA_CLEANED
from src.data_loader import load_raw_tables
from src.cleaning import clean_tables, build_analytical_dataset, save_outputs

tables = load_raw_tables(DATA_RAW)
cleaned = clean_tables(tables)
analytical = build_analytical_dataset(cleaned)
save_outputs(cleaned, analytical)
analytical.head()

## 2. Executive KPIs

In [ ]:
order_level = analytical.drop_duplicates('order_id')
kpis = {
    'total_revenue': analytical['revenue'].sum(),
    'total_orders': analytical['order_id'].nunique(),
    'active_customers': analytical['customer_unique_id'].nunique(),
    'aov': analytical.groupby('order_id')['revenue'].sum().mean(),
    'on_time_delivery_pct': 1 - order_level['late_delivery_flag'].mean(),
    'avg_review_score': order_level['review_score'].mean(),
}
kpis

## 3. RFM Segmentation

In [ ]:
from src.rfm import build_rfm

rfm = build_rfm(analytical)
rfm.groupby('segment').agg(customers=('customer_unique_id', 'nunique'), avg_monetary=('monetary', 'mean')).sort_values('customers', ascending=False)

## 4. Cohort Retention

In [ ]:
from src.cohort import build_monthly_cohort

retention = build_monthly_cohort(order_level)
retention.head()

## 5. Visualization Export

In [ ]:
from src.visualizations import save_core_visualizations

save_core_visualizations(analytical)